In [ ]:
# imports
import sys
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path('../src')))

DATA_ROOT = Path('../data')
FIGURES   = Path('../outputs/figures')
FIGURES.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

In [ ]:
# hyperparamètres
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_WORKERS = 2
MEAN = [0.485]
STD  = [0.229]

In [ ]:
# transformations train : resize + augmentation + normalisation
train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

# transformations val/test : resize + normalisation uniquement
eval_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

In [ ]:
# chargement des datasets
train_dataset = datasets.ImageFolder(str(DATA_ROOT / 'train'), transform=train_transforms)
val_dataset   = datasets.ImageFolder(str(DATA_ROOT / 'val'),   transform=eval_transforms)
test_dataset  = datasets.ImageFolder(str(DATA_ROOT / 'test'),  transform=eval_transforms)

print(f"Classes : {train_dataset.classes}")
print(f"Train   : {len(train_dataset)} images")
print(f"Val     : {len(val_dataset)} images")
print(f"Test    : {len(test_dataset)} images")

In [ ]:
# dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# vérification shape d'un batch
imgs, lbls = next(iter(train_loader))
print(f"Shape : {imgs.shape}")  # [32, 1, 224, 224]
print(f"Min / Max : {imgs.min():.3f} / {imgs.max():.3f}")

In [ ]:
# fonction utilitaire pour dénormaliser avant affichage
def denormalize(tensor):
    mean = torch.tensor(MEAN).view(-1, 1, 1)
    std  = torch.tensor(STD).view(-1, 1, 1)
    return (tensor * std + mean).squeeze(0).clamp(0, 1).numpy()

# dataset sans augmentation pour comparaison
base_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
orig_ds = datasets.ImageFolder(str(DATA_ROOT / 'train'), transform=base_transform)
aug_ds  = datasets.ImageFolder(str(DATA_ROOT / 'train'), transform=train_transforms)

# affichage original vs augmenté
n       = 4
indices = np.random.choice(len(orig_ds), n, replace=False)
fig, axes = plt.subplots(2, n, figsize=(n * 3, 6))
fig.suptitle("Original vs Augmenté", fontsize=12)

for col, idx in enumerate(indices):
    orig_img, label = orig_ds[int(idx)]
    aug_img, _      = aug_ds[int(idx)]
    axes[0, col].imshow(denormalize(orig_img), cmap='gray')
    axes[0, col].set_title(orig_ds.classes[label], fontsize=9)
    axes[0, col].axis('off')
    axes[1, col].imshow(denormalize(aug_img), cmap='gray')
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig(FIGURES / 'original_vs_augmented.png', dpi=150)
plt.show()

In [ ]:
# visualisation d'un batch augmenté (12 images)
imgs, lbls = next(iter(train_loader))
n, cols = 12, 4
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
fig.suptitle("Batch augmenté — Train", fontsize=12)

for i, ax in enumerate(axes.flat):
    if i < n:
        ax.imshow(denormalize(imgs[i]), cmap='gray')
        label = train_dataset.classes[lbls[i]]
        color = '#d62728' if label == 'PNEUMONIA' else '#2ca02c'
        ax.set_title(label, fontsize=9, color=color, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig(FIGURES / 'augmented_batch.png', dpi=150)
plt.show()

In [ ]:
# distribution des pixels après normalisation
imgs_batch, _ = next(iter(train_loader))
m = imgs_batch.mean().item()
s = imgs_batch.std().item()
print(f"Moyenne : {m:.4f} | Ecart-type : {s:.4f}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(imgs_batch.numpy().flatten(), bins=80, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(m, color='red', linestyle='--', label=f'mean = {m:.3f}')
ax.set_xlabel("Valeur pixel (normalisée)")
ax.set_ylabel("Fréquence")
ax.set_title("Distribution des pixels")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'pixel_distribution.png', dpi=150)
plt.show()